In [ ]:
import numpy as np
import plotly.graph_objects as go
import torch

In [ ]:
def visualize_loss_3d(
    loss_class,
    title,
    x_range=(0.1, 10),
    y_range=(0.1, 10),
    steps=50,
    minus_1=False,
    **kwargs,
):
    """
    Args:
        loss_class: The class of the loss function (e.g., QLoss)
        title: Title of the plot
        x_range: Tuple (min, max) for the Target (Ground Truth)
        y_range: Tuple (min, max) for the Input (Prediction)
        kwargs: Arguments to pass to the loss constructor (e.g., min_val=1e-3)
    """

    # 1. Instantiate the Loss
    criterion = loss_class(**kwargs)

    # 2. Create the Grid
    # We use numpy for grid generation, then convert to torch
    targets = np.linspace(x_range[0], x_range[1], steps)
    inputs = np.linspace(y_range[0], y_range[1], steps)

    T, I = np.meshgrid(targets, inputs)

    # Flatten to pass through the PyTorch loss function in batch
    t_flat = torch.from_numpy(T.flatten()).float().reshape(-1, 1)
    i_flat = torch.from_numpy(I.flatten()).float().reshape(-1, 1)

    with torch.no_grad():
        z_flat = criterion(i_flat, t_flat, return_aggregated=False)
        if minus_1:
            z_flat = z_flat - 1.0

    # Reshape back to grid for plotting
    Z = z_flat.numpy().reshape(T.shape)

    fig = go.Figure(data=[go.Surface(z=Z, x=T, y=I)])
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title="Target", yaxis_title="Prediction", zaxis_title="Loss"),
        width=800,
        height=800,
    )
    fig.show()

In [6]:
from training import losses

In [ ]:
visualize_loss_3d(
    losses.QLoss, "QLoss (Q-Error)", x_range=(0, 30), y_range=(0, 30), steps=60
)

In [ ]:
visualize_loss_3d(
    losses.Log1pQLossScaled,
    "Log1pQLossScaled",
    x_range=(-1, 30),
    y_range=(-1, 30),
    steps=60,
    minus_1=True,
)
visualize_loss_3d(
    losses.Log1pQLoss, "Log1pQLoss", x_range=(-1, 30), y_range=(-1, 30), steps=60
)
visualize_loss_3d(
    losses.Log1pHuberQLoss,
    "Log1pHuberQLoss",
    x_range=(-1, 30),
    y_range=(-1, 30),
    steps=60,
)